In [44]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))
from target import build_metabolic_targets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score

from sklearn.inspection import permutation_importance
%matplotlib inline

In [45]:
RF_FEATURES = [
    "energy_kcal", "protein_g", "carb_g", "sugar_g", "fat_total_g",
    "fat_sat_g", "fiber_g", "sodium_mg", "potassium_mg",
    "moderate_rec_min_week",
    "sedentary_min",
]
RF_OPTIONAL_FEATURES = [
    "sleep_hours",
    "vigorous_rec_min_week",
    # "smoked_100_cigs",
    # "avg_drinks_per_day",
]
FEATURES = RF_FEATURES + RF_OPTIONAL_FEATURES


In [46]:
TEST_UIDS_PATH = Path("splits/test_uids.csv")
TRAIN_UIDS_PATH = Path("splits/train_uids.csv")
DATA_PATH = Path("data/processed/nhanes_clean.csv")

df = pd.read_csv(DATA_PATH)
test_uid = pd.read_csv(TEST_UIDS_PATH)
train_uid = pd.read_csv(TRAIN_UIDS_PATH)

df = df.dropna(subset=["metabolic_syndrome"])
df = df.dropna(subset=FEATURES)

In [47]:
test_split = df[df["uid"].isin(test_uid["uid"])]
train_split = df[df["uid"].isin(train_uid["uid"])]
x_train = train_split[FEATURES]
y_train = train_split["metabolic_syndrome"]
x_test = test_split[FEATURES]
y_test = test_split["metabolic_syndrome"]

In [48]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=50 ,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
)
rf.fit(x_train, y_train)

# Performance
y_train_pred = rf.predict(x_train)
y_test_pred = rf.predict(x_test)
print("Training Performance")
print(classification_report(y_train, y_train_pred))
print("Testing Performance")
print(classification_report(y_test, y_test_pred))

y_prob_train = rf.predict_proba(x_train)[:,1]
y_prob_test = rf.predict_proba(x_test)[:,1]
print(
    "Train AUC:",
    roc_auc_score(y_train, y_prob_train)
)
print(
    "Test AUC:",
    roc_auc_score(y_test, y_prob_test)
)

print("Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_test_pred,
))

Training Performance
              precision    recall  f1-score   support

         0.0       0.85      0.58      0.69      5962
         1.0       0.50      0.81      0.62      3079

    accuracy                           0.66      9041
   macro avg       0.68      0.69      0.65      9041
weighted avg       0.73      0.66      0.67      9041

Testing Performance
              precision    recall  f1-score   support

         0.0       0.76      0.50      0.60      1491
         1.0       0.42      0.70      0.53       776

    accuracy                           0.57      2267
   macro avg       0.59      0.60      0.56      2267
weighted avg       0.65      0.57      0.58      2267

Train AUC: 0.7851929275146186
Test AUC: 0.6318149446507222
Confusion Matrix:
[[741 750]
 [229 547]]


In [50]:
perm = permutation_importance(
    rf,
    x_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=42,
)

importance = pd.DataFrame({
    "Feature": x_test.columns,
    "Importance": perm.importances_mean,
    "std": perm.importances_std
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance = importance[
    ~importance["Feature"].isin(["age", "sex"])
]

importance.head(5)


,Feature,Importance,std
12,vigorous_rec_min_week,0.076555,0.009574
0,energy_kcal,0.013569,0.005715
9,moderate_rec_min_week,0.006927,0.008411
6,fiber_g,0.004570,0.003320
7,sodium_mg,0.003717,0.001663
